<a href="https://colab.research.google.com/github/mickePY/codex/blob/main/06_recommandations_banque.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

yves ebanda

# 📊 Recommandations et restitution — Crédit Azur (secteur banque)

> ✅ **Notebook complété** · chiffres calculés dans le code, recommandations et discours rédigés.

**Objectifs**
- Formuler des recommandations opérationnelles structurées
- Intégrer les indicateurs ESG, les limites et les retours utilisateurs
- Préparer 15 minutes de restitution adaptées à chaque interlocuteur

**Sommaire** : 1 Chiffres clés (+ tests) · 2 Recommandations · 3 Indicateurs ESG · 4 Adapter le discours · 5 Plan des 15 minutes · Check-list

**Grille d'évaluation (20 points)** : C2.6 Analyser · C2.7 Visualiser · C2.8 Biais · C2.9 Restituer · C2.10 Recommander, 4 points chacune. Ce notebook couvre surtout C2.6, C2.8, C2.9 et C2.10.

In [ ]:
# ⚙️ À exécuter en premier : récupère la configuration du cours (quelques secondes)
import os, sys
CONFIG_ID = "1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r"

# Suppression du fichier corrompu s'il existe
if os.path.exists("config_projet.py"):
    os.remove("config_projet.py")

# Téléchargement propre avec gdown pour contourner l'avertissement de virus Google Drive
!gdown --id {CONFIG_ID} -O config_projet.py

print("✅ Configuration prête : chaque fichier de données se télécharge au premier usage")

In [ ]:
# Choisissez votre secteur : "commerce", "banque" ou "hotellerie"
SECTEUR = "banque"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config_projet import charger, reference, nettoyage_minimal

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
brut, cfg = charger(SECTEUR)
print(cfg["nom"], "·", brut.shape[0], "lignes,", brut.shape[1], "colonnes")

## 1. Les chiffres clés de votre étude
Les **cinq chiffres** que le groupe cite à l'oral, calculés ci-dessous (aucun chiffre n'est tapé à la main), puis les **tests** qui les rendent solides.

In [ ]:
df = nettoyage_minimal(brut, cfg)
df["annee"] = df[cfg["date"]].dt.year
df["apres_v2"] = df[cfg["date"]] >= pd.Timestamp(cfg["evenement"][0])      # score V2 : 1er mars 2024
av, ap = df[~df["apres_v2"]], df[df["apres_v2"]]
dec = cfg["decision"]                                                       # credit_accorde (0/1)

pnb = df.groupby("annee")[cfg["ca"]].sum()
tx_age = df.groupby(["tranche_age", "apres_v2"])[dec].mean().unstack()
tx_age_tot = df.groupby("tranche_age")[dec].mean()
tx_genre = df.groupby("genre")[dec].mean()
incident = 1 - df[df[dec] == 1].groupby("tranche_age")["rembourse_sans_incident"].mean()   # impayés : seulement sur les crédits accordés

chiffres = pd.DataFrame([
    ("1. PNB estimé 2025",                  f"{pnb[2025]/1e6:.2f} M€ ({pnb[2025]/pnb[2024]-1:+.1%} vs 2024, {pnb[2025]/pnb[2023]-1:+.1%} vs 2023)"),
    ("2. Crédits accordés avant → après V2", f"{av[dec].mean():.1%} → {ap[dec].mean():.1%}"),
    ("3. 18-24 ans avant → après V2",        f"{tx_age.loc['18-24', False]:.1%} → {tx_age.loc['18-24', True]:.1%} (ratio des 4/5 global : {tx_age_tot['18-24']/tx_age_tot.max():.2f})"),
    ("4. Écart femmes / hommes",             f"{tx_genre['F']:.1%} vs {tx_genre['H']:.1%} (ratio {tx_genre['F']/tx_genre['H']:.2f})"),
    ("5. Incidents de paiement 18-24 vs 65+", f"{incident['18-24']:.1%} vs {incident['65+']:.1%} (sur crédits accordés)"),
], columns=["Chiffre", "Valeur"])
chiffres

In [ ]:
# --- Les tests : l'écart avant/après V2 est-il dû au hasard ?
from scipy import stats
import statsmodels.formula.api as smf
import warnings; warnings.filterwarnings("ignore")

# 1) Taux d'accord avant/après : test du khi² sur le tableau croisé
p_chi2 = stats.chi2_contingency(pd.crosstab(df["apres_v2"], df[dec]))[1]
# 2) Score avant/après : Mann-Whitney (ne suppose pas une loi normale)
p_mw = stats.mannwhitneyu(av["score_algo"], ap["score_algo"]).pvalue
print(f"Khi² accord avant/après V2 : p = {p_chi2:.2e}  → différence {'significative' if p_chi2 < 0.05 else 'non significative'}")
print(f"Mann-Whitney score avant/après : p = {p_mw:.2e}")
print(f"Score moyen : {av['score_algo'].mean():.3f} → {ap['score_algo'].mean():.3f} ; seuil d'accord observé = {df.loc[df[dec]==1, 'score_algo'].min():.2f}")

# 3) Le genre a-t-il un effet DIRECT une fois âge, contrat, région, endettement et montant contrôlés ?
m = smf.logit(f"{dec} ~ C(genre) + C(tranche_age) + C(contrat) + C(region) + taux_endettement + np.log(montant_demande)",
              data=df.dropna(subset=["revenu_mensuel"])).fit(disp=0)
print(f"Régression logistique — coefficient « genre = H » : {m.params['C(genre)[T.H]']:.3f} (p = {m.pvalues['C(genre)[T.H]']:.2f})")
print(f"Même régression — coefficient « contrat = CDI » : {m.params['C(contrat)[T.CDI]']:.2f} (p < 0,001) ; « âge 45-54 » vs 18-24 : {m.params['C(tranche_age)[T.45-54]']:.2f}")

# 4) Saisonnalité : indice mensuel moyen (1 = un mois moyen)
mens = df.set_index(cfg["date"])[cfg["ca"]].resample("MS").sum()
indice = (mens.groupby(mens.index.month).mean() / mens.mean()).round(2)
print("Indice saisonnier du PNB (mois 1 à 12) :", indice.to_dict())

**Lecture des tests** — La baisse des accords et du score est **statistiquement certaine** (p < 0,001) : ce n'est pas du bruit. À l'inverse, **le genre n'a pas d'effet direct significatif** une fois l'âge, le contrat, l'endettement et le montant contrôlés, alors que le **contrat (CDI)** et l'**âge** pèsent très lourd. Conclusion prudente : l'écart F/H passe surtout par des variables corrélées au genre (revenu plus bas, score plus bas à âge égal), c'est un **risque de proxy**, pas une preuve de discrimination directe. Saisonnalité : creux en **août** et **décembre** (indice ≈ 0,73), pic en **mai** (≈ 1,17).

## 2. Vos recommandations
Cinq recommandations, chacune sur le même modèle (constat chiffré → action SMART → impact → indicateur → limite).

| # | Constat chiffré | Recommandation (SMART) | Impact attendu | Indicateur de suivi | Limite ou condition |
|---|---|---|---|---|---|
| 1 | Crédits accordés **76 % → 57 %** après la V2 alors que le seuil est inchangé (0,66) ; PNB mensuel moyen **346 k€ → 264 k€** (2023 → 2025) | **Auditer puis recalibrer le score V2** avec backtest sur les données d'avant V2, **avant le 31/03/2027**, responsable : équipe data + risque | Retrouver un taux d'accord d'au moins 65 % sans dégrader le risque ; l'ordre de grandeur du gain est limité par la limite ci-contre | Taux d'accord et PNB mensuel (mensuel), taux d'incident (trimestriel) | On ne connaît **pas** le remboursement des dossiers refusés (biais de sélection) : le gain n'est pas chiffrable sérieusement avant un test |
| 2 | **18-24 ans : 18 % d'accords** (ratio 4/5 = 0,21) contre 87 % pour les 55-64 ; pourtant incidents 4,9 % contre 1,6 % pour les 65+ | **Revue humaine pilote** des dossiers 18-34 ans et CDD dont le score est entre **0,55 et 0,66** (666 dossiers 18-34 ans refusés depuis la V2 avec un score d'au moins 0,55), lancée au 2e trimestre 2027 sur 6 mois | Rouvrir l'accès au crédit à des profils jugés trop sévèrement, avec un PNB moyen de 817 € par crédit accordé | Ratio 4/5 par âge (cible ≥ 0,5 à 12 mois), taux d'incident du pilote ≤ 3 % | Le CDD est un vrai facteur de risque ; mesurer le pilote avant de généraliser ; base de calcul : 2720 crédits accordés avec un score < 0,75 ont 4,6 % d'incidents |
| 3 | Femmes 60 % d'accords contre 67 % pour les hommes (ratio 0,89) ; revenu médian 2190 € contre 2525 € ; **genre non significatif** après contrôles | **Test de proxy** : refaire le score sans revenu ni contrat et comparer ; rapport au comité **avant le 30/06/2027** | Éviter une discrimination indirecte, réduire le risque juridique et de réputation | Ratio F/H (cible ≥ 0,9), écart de score à profil égal (cible < 0,02) | Un écart de score peut refléter un risque réel ; ne pas supprimer une variable sans mesurer la perte de pouvoir prédictif |
| 4 | **65+ : 7,0 % des demandes** contre 26 % en référence ; **Outre-mer** : 207 dossiers (1,0 %) | Suivre le ratio avec `reference_population.csv`, **objectif ≥ 0,8** à fin 2027 pour 65+ et Outre-mer ; actions commerciales ciblées | Modèle mieux entraîné sur tous les publics, nouveaux clients | Ratio part observée ÷ part de référence, par groupe (trimestriel) | La part des demandes dépend aussi du besoin ; effectifs Outre-mer trop faibles pour conclure (5,9 % d'incidents sur 119 crédits) |
| 5 | **578 revenus manquants (2,8 %)**, **82 doublons**, 11 âges < 18 ans dans le fichier brut | **Contrôles de saisie obligatoires** (revenu, âge ≥ 18) et dédoublonnage automatique **dès le 1er trimestre 2027** | Fiabilité du score et des tableaux de bord | Taux de complétude ≥ 99 %, doublons = 0 | Impact direct faible sur le PNB, mais condition de confiance pour les autres actions |

**Pourquoi ces recommandations sont « SMART »** : *Spécifique* (une action nommée), *Mesurable* (un indicateur et une cible), *Atteignable* (pilote à l'échelle d'une équipe), *Réaliste* (limite explicite), *Temporel* (date butoir). **Honnêteté sur les limites** : aucun chiffre d'impact en euros n'est annoncé pour les recommandations 1 et 2, car le risque réel des dossiers refusés est inconnu ; le pilote de la recommandation 2 sert justement à le mesurer.

## 3. Les indicateurs ESG
Quels indicateurs environnementaux, sociaux ou de gouvernance suivre, au vu de l'analyse ?

| Pilier | Indicateur à suivre | Pourquoi (lien avec l'analyse) | Fréquence |
|---|---|---|---|
| **S** — Social (inclusion) | Ratio des 4/5 du taux d'accord par âge, genre et région | 18-24 ans à 0,21 ; Outre-mer à 0,80 | Trimestrielle |
| **S** | Part des 18-34 ans financés et taux d'incident associé | Mesurer l'inclusion **et** le risque | Trimestrielle |
| **S** | Nombre de dossiers refusés ayant eu une revue humaine, et issue | Droit à un second regard | Mensuelle |
| **G** — Gouvernance | Date du dernier audit du score et de ses variables, comité de suivi | Le score V2 a fait chuter les accords sans revue connue | Annuelle |
| **G** | Part des décisions automatisées expliquées au client | Obligations liées à la décision automatisée (RGPD, art. 22) et au scoring de crédit, classé à haut risque par le règlement européen sur l'IA : **à vérifier avec le DPO** | Annuelle |
| **G** | Complétude et qualité des données (revenus manquants, doublons) | 2,8 % revenus manquants, 82 doublons | Mensuelle |
| **E** — Environnement | Part des demandes traitées en ligne (web : 39 % des dossiers) | Moins de papier et de déplacements ; **l'empreinte carbone n'est pas mesurable avec ce fichier** | Annuelle |
| **E** | Part des crédits travaux fléchés vers la rénovation énergétique | Champ **à créer** : la colonne `type_credit = travaux` ne le précise pas | Annuelle |

## 4. Adapter le discours
Un même résultat, trois publics.

| Interlocuteur | Ce qu'il veut savoir | Ce que vous lui montrez |
|---|---|---|
| **Direction métier** | Où perd-on de l'activité, sur quels produits et quels clients, et que faire concrètement ? | La courbe du PNB avec la rupture de mars 2024, le tableau avant/après V2 par âge et par contrat, le pilote de revue humaine (reco 2) |
| **Décideurs (comité de direction)** | Combien, quand, quel risque, quelle décision attendue ? | Les trois chiffres du tableau de bord (PNB 2025 : -5 % vs 2024, accords 76 % → 57 %), les 5 recommandations avec dates, la décision : **valider l'audit du score V2 avant fin du 1er trimestre 2027** |
| **Conformité (DPO, référent éthique)** | Y a-t-il discrimination, sur quelles données, avec quelles preuves et quelles limites ? | Le ratio des 4/5 par groupe, la distinction **écart constaté / discrimination démontrée**, le test de proxy (reco 3), le biais de sélection, les indicateurs ESG, les limites de la référence de population |

**Règle pour parler des biais sans jargon** : *qui* (les 18-24 ans, les CDD), *de combien* (18 % d'accords contre 87 % pour les 55-64 ans), *pourquoi* (le score V2 donne moins de points à ces profils, alors que leurs incidents de paiement restent faibles). Éviter « disparate impact » : dire « un groupe est nettement moins souvent accepté ».

## 5. Le plan des 15 minutes
Situation → complication → résolution, puis 5 minutes de questions. **Chaque membre du groupe prend la parole.** Le dashboard HTML sert de support (pas de diapositives obligatoires) ; le PDF fourni suit le même plan.

| Temps | Contenu | Support (dashboard / PDF) | Qui (à adapter à la taille du groupe) |
|---|---|---|---|
| 1 min | **Contexte** : Crédit Azur, 4 types de crédit, 20 384 demandes en 3 ans, question du commanditaire : pourquoi l'activité baisse et à qui profite la décision | Titre + essentiel en 30 secondes | Membre 1 |
| 3 min | **Tendance** : saisonnalité (creux août/décembre), rupture au score V2 (1er mars 2024), tests | Courbe du PNB + tableau historique | Membre 2 |
| 3 min | **Segments** : tous les produits baissent (auto et travaux portent 84 % du PNB) ; le canal n'explique rien ; l'âge et le contrat, si | Courbe par type de crédit (menu) + avant/après V2 | Membre 3 |
| 3 min | **Biais** : qui (18-24, 25-34, CDD), de combien (ratio 0,21), pourquoi (score V2, proxy revenu/contrat) ; limites | Parité + alertes écrites | Membre 4 |
| 4 min | **Recommandations** : 5 actions, indicateurs, limites, ESG | Tableau des recommandations | Membre 1 et 2 |
| 1 min | **Décision attendue** : valider l'audit du score V2 avant fin T1 2027 | Dernier encart | Membre 3 |

*Puis 5 minutes de questions.* **Questions probables du jury** : « Pourquoi le seuil inchangé est-il important ? » (parce que seul le score a changé) · « Le genre est-il discriminé ? » (écart constaté, pas d'effet direct démontré, risque de proxy) · « Quel gain en euros ? » (non chiffrable sans pilote, biais de sélection) · « Votre référence de population est-elle fiable ? » (fichier du cours, part_reference ; ordres de grandeur pédagogiques, pas des chiffres INSEE exacts).

## ✅ Avant de passer
- [ ] Le dashboard HTML s'ouvre sur le poste de présentation (tester hors connexion).
- [ ] Chaque recommandation a un chiffre, un indicateur et une limite. ✔ (voir section 2)
- [ ] Les biais sont présentés sans jargon : qui est concerné, de combien, pourquoi. ✔ (voir section 4)
- [ ] La dernière version du notebook est enregistrée dans GitHub.